# Baseline CNNs on Tiny-ImageNet — FP32 & QAT (INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare the 4 Phase 1 reference architectures in FP32, then apply
Quantization-Aware Training (QAT) to the QAT-compatible models to obtain INT8 versions.

| Model | QAT | Notes |
|---|---|---|
| AlexNetTV | ✓ | Pretrained torchvision AlexNet, Conv-ReLU fused |
| VGGStyleCNN | ✓ | From-scratch, stacked 3×3, Conv-BN-ReLU stages |
| ResNet18TV | ✗ | Pretrained ResNet-18; residual adds lack FloatFunctional → FP32 only |
| MobileNetV2TV | ✗ | Pretrained MobileNetV2; inverted residual adds → FP32 only |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training (AlexNetTV, VGGStyleCNN)
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Imports & reproducibility


In [1]:
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, load_best_model, convert_to_int8,
    build_comparison_table, create_results_summary, disk_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models import (
    build_alexnet, AlexNetTV,
    build_vggstylecnn, VGGStyleCNN,
    build_resnet18, ResNet18TV,
    build_mobilenetv2, MobileNetV2TV,
)

torch.backends.quantized.engine = "fbgemm"

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "baselines_qat_phase1"
RESULTS_DIR = project_root / "results" / "baselines_qat_phase1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

cuda


## 2. Dataset & loaders


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train
Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model shape check


In [4]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetTV",    build_alexnet),
    ("VGGStyleCNN",  build_vggstylecnn),
    ("ResNet18TV",   build_resnet18),
    ("MobileNetV2TV",build_mobilenetv2),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{label:20s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

AlexNetTV            OK -> (2, 200), params=57.82M
VGGStyleCNN          OK -> (2, 200), params=2.41M
ResNet18TV           OK -> (2, 200), params=11.28M
MobileNetV2TV        OK -> (2, 200), params=2.48M


## 4. Model registration

Fuse maps are index-based paths inside `features` for flat-Sequential models.
`ResNet18TV` and `MobileNetV2TV` are registered but excluded from QAT — their residual
adds use plain `+` / inverted bottleneck without `FloatFunctional`, so fake-quant observers
cannot intercept them.

In [5]:
# Conv-ReLU (torchvision AlexNet has no BN in features)
FUSE_MAP_ALEXNET_TV = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

# Conv-BN-ReLU, two per VGG stage (5 stages × 2 = 10 groups)
FUSE_MAP_VGG = [
    ["0","1","2"],["3","4","5"],
    ["7","8","9"],["10","11","12"],
    ["14","15","16"],["17","18","19"],
    ["21","22","23"],["24","25","26"],
    ["28","29","30"],["31","32","33"],
]

MODEL_REGISTRY.clear()
register_model("alexnet_tv",  build_alexnet,     fuse_map=FUSE_MAP_ALEXNET_TV, fuse_root_attr="features", lr=3e-4)
register_model("vgg_style",   build_vggstylecnn, fuse_map=FUSE_MAP_VGG,        fuse_root_attr="features", lr=1e-3)
register_model("resnet18_tv", build_resnet18,    fuse_map=[],                                             lr=1e-4)  # ponytail: QAT skipped — vanilla residual adds
register_model("mobilenetv2", build_mobilenetv2, fuse_map=[],                                             lr=1e-4)  # ponytail: QAT skipped — inverted residual adds

print("Registered:", list(MODEL_REGISTRY))

Registered: ['alexnet_tv', 'vgg_style', 'resnet18_tv', 'mobilenetv2']


## 5. FP32 training


In [6]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    if (SAVE_DIR / f"{name}_best.pth").exists():
        print(f"Skipping FP32 training for {name} (checkpoint found).")
        continue
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-baselines",
        group="fp32-baselines",
        name=f"{name}_fp32",
        config={**asdict(model_cfg), "arch": name, "phase": "fp32",
                "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
                "dataset": "tiny-imagenet-200"},
        tags=["phase1", "baselines", "tiny-imagenet", "fp32"],
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
        log_file=SAVE_DIR / f"{name}.log",
    )
    results = trainer.fit()
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

Training: alexnet_tv  lr=0.0003  epochs=2


Validation: 100%|██████████| 157/157 [00:01<00:00, 148.75it/s, loss=4.2562, top1=12.95%, top5=35.51%]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/2 | train_loss=4.6435 train_acc=7.93% | val_loss=4.2562 val_acc=12.95% val_top5=35.51% | lr=1.50e-04 peak_mem=1768MB time=47.2s
Validation: 100%|██████████| 157/157 [00:01<00:00, 151.09it/s, loss=3.9049, top1=19.76%, top5=45.51%]
Epoch   2/2 | train_loss=4.2381 train_acc=14.24% | val_loss=3.9049 val_acc=19.76% val_top5=45.51% | lr=0.00e+00 peak_mem=1409MB time=47.0s

================= Run Summary =================
Model          : alexnet_tv
Epochs         : 2
Best Val Top-1 : 19.76%
Best Val Top-5 : 45.51%
Final Val Top-1: 19.76%
Final Val Top-5: 45.51%
Best Val Loss  : 3.9049
Total Time     : 00:01:35


best_val_acc,▁█
epoch_time_s,█▁
lr,█▁
peak_gpu_mem_mb,█▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_top5,▁█
best_val_acc,19.76
epoch_time_s,47.02963


Training: vgg_style  lr=0.001  epochs=2


Validation: 100%|██████████| 157/157 [00:01<00:00, 109.13it/s, loss=4.9569, top1=4.74%, top5=15.89%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/2 | train_loss=5.0092 train_acc=2.98% | val_loss=4.9569 val_acc=4.74% val_top5=15.89% | lr=5.00e-04 peak_mem=888MB time=24.3s
Validation: 100%|██████████| 157/157 [00:01<00:00, 110.57it/s, loss=4.4039, top1=10.59%, top5=30.33%]
Epoch   2/2 | train_loss=4.5434 train_acc=8.36% | val_loss=4.4039 val_acc=10.59% val_top5=30.33% | lr=0.00e+00 peak_mem=503MB time=24.4s

================= Run Summary =================
Model          : vgg_style
Epochs         : 2
Best Val Top-1 : 10.59%
Best Val Top-5 : 30.33%
Final Val Top-1: 10.59%
Final Val Top-5: 30.33%
Best Val Loss  : 4.4039
Total Time     : 00:00:48


...

best_val_acc,▁█
epoch_time_s,█▁
lr,█▁
peak_gpu_mem_mb,█▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_top5,▁█
best_val_acc,46.94
epoch_time_s,24.72801


Training: mobilenetv2  lr=0.0001  epochs=2


Validation: 100%|██████████| 157/157 [00:01<00:00, 113.61it/s, loss=2.9790, top1=41.90%, top5=70.05%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/2 | train_loss=3.9715 train_acc=21.46% | val_loss=2.9790 val_acc=41.90% val_top5=70.05% | lr=5.00e-05 peak_mem=1291MB time=26.2s
Validation: 100%|██████████| 157/157 [00:01<00:00, 129.93it/s, loss=2.8239, top1=45.83%, top5=72.93%]
Epoch   2/2 | train_loss=3.2452 train_acc=35.16% | val_loss=2.8239 val_acc=45.83% val_top5=72.93% | lr=0.00e+00 peak_mem=540MB time=25.2s

================= Run Summary =================
Model          : mobilenetv2
Epochs         : 2
Best Val Top-1 : 45.83%
Best Val Top-5 : 72.93%
Final Val Top-1: 45.83%
Final Val Top-5: 72.93%
Best Val Loss  : 2.8239
Total Time     : 00:00:51


best_val_acc,▁█
epoch_time_s,█▁
lr,█▁
peak_gpu_mem_mb,█▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_top5,▁█
best_val_acc,45.83
epoch_time_s,25.24342



FP32 training complete.


## 6. Quantization-Aware Training (QAT)

`ResNet18TV` and `MobileNetV2TV` are excluded: their residual adds (`+` / inverted bottleneck)
aren't interceptable by fake-quant observers without `FloatFunctional`.
QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [7]:
QAT_SKIP = {"resnet18_tv", "mobilenetv2"}  # ponytail: no FloatFunctional residual adds

qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    if name in QAT_SKIP:
        print(f"Skipping QAT for {name} (QAT_SKIP).")
        continue

    if (SAVE_DIR / f"qat_{name}_best.pth").exists():
        print(f"Skipping QAT for {name} (checkpoint found).")
        continue

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-baselines",
        group="qat-baselines",
        name=f"{name}_qat",
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase1", "baselines", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit()
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

QAT fine-tuning: alexnet_tv


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Validation: 100%|██████████| 157/157 [00:01<00:00, 94.10it/s, loss=3.7931, top1=23.39%, top5=48.71%] 
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/2 | train_loss=4.0701 train_acc=17.39% | val_loss=3.7931 val_acc=23.39% val_top5=48.71% | lr=5.00e-06 peak_mem=1410MB time=51.6s
Validation: 100%|██████████| 157/157 [00:01<00:00, 94.42it/s, loss=3.7722, top1=23.81%, top5=49.35%] 
Epoch   2/2 | train_loss=4.0263 train_acc=18.22% | val_loss=3.7722 val_acc=23.81% val_top5=49.35% | lr=0.00e+00 peak_mem=1410MB time=51.4s

================= Run Summary =================
Model          : qat_alexnet_tv
Epochs         : 2
Best Val Top-1 : 23.81%
Best Val Top-5 : 49.35%
Final Val Top-1: 23.81%
Final Val Top-5: 49.35%
Best Val Loss  : 3.7722
Total Time     : 00:01:44


best_val_acc,▁█
epoch_time_s,█▁
lr,█▁
peak_gpu_mem_mb,█▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_top5,▁█
best_val_acc,23.81
epoch_time_s,51.39478


QAT fine-tuning: vgg_style


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Validation: 100%|██████████| 157/157 [00:01<00:00, 81.65it/s, loss=4.1682, top1=14.17%, top5=36.90%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/2 | train_loss=4.3281 train_acc=11.77% | val_loss=4.1682 val_acc=14.17% val_top5=36.90% | lr=5.00e-06 peak_mem=1086MB time=39.8s
Validation: 100%|██████████| 157/157 [00:02<00:00, 77.68it/s, loss=4.1578, top1=14.18%, top5=37.04%]
Epoch   2/2 | train_loss=4.3002 train_acc=12.29% | val_loss=4.1578 val_acc=14.18% val_top5=37.04% | lr=0.00e+00 peak_mem=708MB time=39.9s

================= Run Summary =================
Model          : qat_vgg_style
Epochs         : 2
Best Val Top-1 : 14.18%
Best Val Top-5 : 37.04%
Final Val Top-1: 14.18%
Final Val Top-5: 37.04%
Best Val Loss  : 4.1578
Total Time     : 00:01:19


best_val_acc,▁█
epoch_time_s,▁█
lr,█▁
peak_gpu_mem_mb,█▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_top5,▁█
best_val_acc,14.18
epoch_time_s,39.86934


Skipping QAT for resnet18_tv (QAT_SKIP).
Skipping QAT for mobilenetv2 (QAT_SKIP).

QAT training complete.


## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.


In [8]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

INT8 conversion done.
alexnet_tv             | loss=3.4385 | top1=23.49% | top5=49.19%
vgg_style              | loss=3.9057 | top1=14.47% | top5=37.02%


## 8. FP32 evaluation — reload best checkpoints


In [9]:
CTORS = {
    "alexnet_tv":  build_alexnet,
    "vgg_style":   build_vggstylecnn,
    "resnet18_tv": build_resnet18,
    "mobilenetv2": build_mobilenetv2,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

alexnet_tv             | loss=3.5998 | top1=19.63% | top5=45.43%
vgg_style              | loss=4.1601 | top1=10.63% | top5=30.42%
resnet18_tv            | loss=2.2670 | top1=46.95% | top5=73.50%
mobilenetv2            | loss=2.3409 | top1=45.82% | top5=72.97%


## 9. Final comparison table


In [10]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,resnet18_tv,FP32,46.950695,73.501438,2.267025,11.279112,129.208559
1,mobilenetv2,FP32,45.822307,72.971761,2.340923,2.480072,28.751802
2,alexnet_tv,FP32,19.627163,45.426965,3.599760,57.823240,661.753958
3,vgg_style,FP32,10.632294,30.417612,4.160062,2.405288,27.584690
4,alexnet_tv_INT8,INT8,23.493534,49.188855,3.438466,57.823240,55.332613
5,vgg_style_INT8,INT8,14.468633,37.022009,3.905734,2.405288,2.376703


In [11]:
import json as _json

# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:22s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:22s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_best.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:22s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in fp32_best:
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(SAVE_DIR / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
    )
    out = RESULTS_DIR / f"{name}_summary.json"
    with open(out, "w") as f:
        _json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

alexnet_tv             FP32 | 0.096 ms/img
vgg_style              FP32 | 0.110 ms/img
resnet18_tv            FP32 | 0.110 ms/img
mobilenetv2            FP32 | 0.115 ms/img
alexnet_tv             INT8 | 0.621 ms/img
vgg_style              INT8 | 1.173 ms/img
alexnet_tv             | 0.095 GMACs
vgg_style              | 0.231 GMACs
resnet18_tv            | 0.149 GMACs
mobilenetv2            | 0.026 GMACs
Saved: alexnet_tv_summary.json
Saved: vgg_style_summary.json
Saved: resnet18_tv_summary.json
Saved: mobilenetv2_summary.json


In [12]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:26s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

RANKING BY TOP-1 ACCURACY (all precisions)
 1. resnet18_tv                [FP32] | top1= 46.95% | top5= 73.50% | params= 11.28M | size=129.21MB
 2. mobilenetv2                [FP32] | top1= 45.82% | top5= 72.97% | params=  2.48M | size= 28.75MB
 3. alexnet_tv_INT8            [INT8] | top1= 23.49% | top5= 49.19% | params= 57.82M | size= 55.33MB
 4. alexnet_tv                 [FP32] | top1= 19.63% | top5= 45.43% | params= 57.82M | size=661.75MB
 5. vgg_style_INT8             [INT8] | top1= 14.47% | top5= 37.02% | params=  2.41M | size=  2.38MB
 6. vgg_style                  [FP32] | top1= 10.63% | top5= 30.42% | params=  2.41M | size= 27.58MB


## 10. Persist experiment summary


In [13]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-baselines`** — one run per architecture, FP32 training
- **`qat-baselines`** — one run per architecture (excl. `resnet18_tv`), QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```
